In [0]:
# 02_ingest — ADLS raw/ JSONL → Bronze Delta 적재

In [0]:
dbutils.widgets.text("catalog", "access_drift_dev")
dbutils.widgets.text("storage_account", "acdriftadls")
dbutils.widgets.text("raw_base", "abfss://lakehouse@acdriftadls.dfs.core.windows.net/raw/dev")

CATALOG = dbutils.widgets.get("catalog")
STORAGE = dbutils.widgets.get("storage_account")
RAW_BASE_PARAM = dbutils.widgets.get("raw_base")
RAW_BASE = RAW_BASE_PARAM or f"abfss://lakehouse@{STORAGE}.dfs.core.windows.net/raw"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

SCHEMA_VERSION = "0.1.0"

def _get_run_id():
    try:
        return dbutils.notebook.entry_point.getDbutils().notebook().getContext().currentRunId().get()
    except Exception:
        return None

def add_meta(df, src_system, src_object, file_path):
    src_cols = df.columns  # 메타 추가 전 원본 컬럼 목록 캡처
    raw_payload = F.to_json(F.struct(*src_cols))
    return (
        df
        .withColumn("_src_system",     F.lit(src_system))
        .withColumn("_src_object",     F.lit(src_object))
        .withColumn("_src_file_path",  F.lit(file_path))
        .withColumn("_ingestion_ts",   F.current_timestamp())
        .withColumn("_run_id",         F.lit(_get_run_id()))
        .withColumn("_batch_id",       F.date_format(F.current_timestamp(), "yyyyMMddHHmmss"))
        .withColumn("_raw_payload",    raw_payload)
        .withColumn("_checksum",       F.md5(raw_payload))
        .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    )

## azure/applications → bronze_azure_applications

In [0]:
path = f"{RAW_BASE}/azure/applications.jsonl"
df = spark.read.json(path)
df = add_meta(df, "entra_id", "applications", path)
df = (df
      .withColumnRenamed("appId", "app_id")
      .withColumnRenamed("displayName", "display_name")
      .withColumn("created_at", F.to_timestamp("createdDateTime"))
      .drop("createdDateTime"))

(df.select(
    "_src_system","_src_object","_src_file_path","_ingestion_ts",
    "_run_id","_batch_id","_checksum","_raw_payload","_schema_version",
    "app_id","display_name","created_at"
).write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.bronze.bronze_azure_applications"))
print(f"bronze_azure_applications: {df.count()} rows")

## azure/application_owners → bronze_azure_application_owners

In [0]:
path = f"{RAW_BASE}/azure/application_owners.jsonl"
df = spark.read.json(path)
df = add_meta(df, "entra_id", "application_owners", path)
df = (df
      .withColumnRenamed("appId", "app_id")
      .withColumnRenamed("userPrincipalName", "user_principal_name"))

(df.select(
    "_src_system","_src_object","_src_file_path","_ingestion_ts",
    "_run_id","_batch_id","_checksum","_raw_payload","_schema_version",
    "app_id","user_principal_name"
).write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.bronze.bronze_azure_application_owners"))
print(f"bronze_azure_application_owners: {df.count()} rows")

## azure/password_credentials → bronze_azure_password_credentials

In [0]:
path = f"{RAW_BASE}/azure/password_credentials.jsonl"
df = spark.read.json(path)
df = add_meta(df, "entra_id", "password_credentials", path)
df = (df
      .withColumnRenamed("appId", "app_id")
      .withColumnRenamed("keyId", "key_id")
      .withColumnRenamed("displayName", "display_name")
      .withColumn("end_date_time", F.to_timestamp("endDateTime"))
      .drop("endDateTime"))

(df.select(
    "_src_system","_src_object","_src_file_path","_ingestion_ts",
    "_run_id","_batch_id","_checksum","_raw_payload","_schema_version",
    "app_id","key_id","display_name","end_date_time","status"
).write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.bronze.bronze_azure_password_credentials"))
print(f"bronze_azure_password_credentials: {df.count()} rows")

## identity/lifecycle_seed → bronze_identity_lifecycle_seed

In [0]:
path = f"{RAW_BASE}/identity/lifecycle_seed.jsonl"
df = spark.read.json(path)
df = add_meta(df, "identity_seed", "lifecycle_seed", path)
df = (df
      .withColumnRenamed("userPrincipalName", "user_principal_name")
      .withColumnRenamed("displayName", "display_name")
      .withColumnRenamed("employmentStatus", "employment_status")
      .withColumn("termination_date", F.to_date("terminationDate"))
      .drop("terminationDate"))

(df.select(
    "_src_system","_src_object","_src_file_path","_ingestion_ts",
    "_run_id","_batch_id","_checksum","_raw_payload","_schema_version",
    "user_principal_name","display_name","employment_status","termination_date"
).write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.bronze.bronze_identity_lifecycle_seed"))
print(f"bronze_identity_lifecycle_seed: {df.count()} rows")

## assets/assets_seed → bronze_assets_seed

In [0]:
path = f"{RAW_BASE}/assets/assets_seed.jsonl"
df = spark.read.json(path)
df = add_meta(df, "assets_seed", "assets_seed", path)
df = df.withColumnRenamed("assetId", "asset_id")

(df.select(
    "_src_system","_src_object","_src_file_path","_ingestion_ts",
    "_run_id","_batch_id","_checksum","_raw_payload","_schema_version",
    "asset_id","name","environment","sensitivity"
).write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.bronze.bronze_assets_seed"))
print(f"bronze_assets_seed: {df.count()} rows")

## access/access_edges_seed → bronze_access_edges_seed

In [0]:
path = f"{RAW_BASE}/access/access_edges_seed.jsonl"
df = spark.read.json(path)
df = add_meta(df, "access_seed", "access_edges_seed", path)
df = (df
      .withColumnRenamed("principalId", "principal_id")
      .withColumnRenamed("assetId", "asset_id"))

if "lastUsedDateTime" in df.columns:
    df = df.withColumn("last_used_at", F.to_timestamp("lastUsedDateTime"))
elif "last_used_at" in df.columns:
    df = df.withColumn("last_used_at", F.to_timestamp("last_used_at"))
else:
    df = df.withColumn("last_used_at", F.lit(None).cast("timestamp"))

(df.select(
    "_src_system","_src_object","_src_file_path","_ingestion_ts",
    "_run_id","_batch_id","_checksum","_raw_payload","_schema_version",
    "principal_id","asset_id","relationship","permission","status","last_used_at"
).write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.bronze.bronze_access_edges_seed"))
print(f"bronze_access_edges_seed: {df.count()} rows")

## Smoke test

## SaaS source-like seed → Bronze


In [ ]:
COMMON_BRONZE_SCHEMA = [
    T.StructField("_src_system", T.StringType(), False),
    T.StructField("_src_object", T.StringType(), False),
    T.StructField("_src_file_path", T.StringType(), True),
    T.StructField("_ingestion_ts", T.TimestampType(), False),
    T.StructField("_run_id", T.StringType(), True),
    T.StructField("_batch_id", T.StringType(), True),
    T.StructField("_checksum", T.StringType(), True),
    T.StructField("_raw_payload", T.StringType(), False),
    T.StructField("_schema_version", T.StringType(), False),
]


def raw_file_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def write_empty_bronze_table(table_name, business_schema):
    schema = T.StructType(COMMON_BRONZE_SCHEMA + business_schema)
    empty_df = spark.createDataFrame([], schema)
    (empty_df.write.format("delta").mode("overwrite")
     .saveAsTable(f"{CATALOG}.bronze.{table_name}"))


def transform_source(df, renames=None, timestamps=None, dates=None):
    for source, target in renames or []:
        df = df.withColumnRenamed(source, target)
    for source, target in timestamps or []:
        df = df.withColumn(target, F.to_timestamp(source)).drop(source)
    for source, target in dates or []:
        df = df.withColumn(target, F.to_date(source)).drop(source)
    return df


SAAS_BRONZE_SOURCES = [
    {
        "path": "github/apps.jsonl",
        "src_system": "github_seed",
        "src_object": "github_apps",
        "table": "bronze_github_apps",
        "renames": [("appId", "app_id"), ("displayName", "display_name"), ("ownerEmail", "owner_email"), ("ownerEmploymentStatus", "owner_employment_status")],
        "dates": [("ownerTerminationDate", "owner_termination_date")],
        "columns": ["app_id", "slug", "display_name", "status", "owner_email", "owner_employment_status", "owner_termination_date"],
        "schema": [
            T.StructField("app_id", T.StringType(), True),
            T.StructField("slug", T.StringType(), True),
            T.StructField("display_name", T.StringType(), True),
            T.StructField("status", T.StringType(), True),
            T.StructField("owner_email", T.StringType(), True),
            T.StructField("owner_employment_status", T.StringType(), True),
            T.StructField("owner_termination_date", T.DateType(), True),
        ],
    },
    {
        "path": "github/credentials.jsonl",
        "src_system": "github_seed",
        "src_object": "github_credentials",
        "table": "bronze_github_credentials",
        "renames": [("appId", "app_id"), ("credentialId", "credential_id"), ("credentialType", "credential_type"), ("displayName", "display_name")],
        "timestamps": [("expiresAt", "expires_at")],
        "columns": ["app_id", "credential_id", "credential_type", "display_name", "status", "expires_at"],
        "schema": [
            T.StructField("app_id", T.StringType(), True),
            T.StructField("credential_id", T.StringType(), True),
            T.StructField("credential_type", T.StringType(), True),
            T.StructField("display_name", T.StringType(), True),
            T.StructField("status", T.StringType(), True),
            T.StructField("expires_at", T.TimestampType(), True),
        ],
    },
    {
        "path": "github/repos.jsonl",
        "src_system": "github_seed",
        "src_object": "github_repos",
        "table": "bronze_github_repos",
        "renames": [("repoId", "repo_id")],
        "columns": ["repo_id", "name", "visibility", "environment", "sensitivity"],
        "schema": [
            T.StructField("repo_id", T.StringType(), True),
            T.StructField("name", T.StringType(), True),
            T.StructField("visibility", T.StringType(), True),
            T.StructField("environment", T.StringType(), True),
            T.StructField("sensitivity", T.StringType(), True),
        ],
    },
    {
        "path": "github/access_edges.jsonl",
        "src_system": "github_seed",
        "src_object": "github_access_edges",
        "table": "bronze_github_access_edges",
        "renames": [("principalId", "principal_id"), ("assetId", "asset_id")],
        "columns": ["principal_id", "asset_id", "relationship", "permission", "status"],
        "schema": [
            T.StructField("principal_id", T.StringType(), True),
            T.StructField("asset_id", T.StringType(), True),
            T.StructField("relationship", T.StringType(), True),
            T.StructField("permission", T.StringType(), True),
            T.StructField("status", T.StringType(), True),
        ],
    },
    {
        "path": "google_workspace/users.jsonl",
        "src_system": "google_workspace_seed",
        "src_object": "google_workspace_users",
        "table": "bronze_google_workspace_users",
        "renames": [("userId", "user_id"), ("displayName", "display_name"), ("employmentStatus", "employment_status")],
        "dates": [("terminationDate", "termination_date")],
        "columns": ["user_id", "email", "display_name", "status", "employment_status", "termination_date"],
        "schema": [
            T.StructField("user_id", T.StringType(), True),
            T.StructField("email", T.StringType(), True),
            T.StructField("display_name", T.StringType(), True),
            T.StructField("status", T.StringType(), True),
            T.StructField("employment_status", T.StringType(), True),
            T.StructField("termination_date", T.DateType(), True),
        ],
    },
    {
        "path": "google_workspace/assets.jsonl",
        "src_system": "google_workspace_seed",
        "src_object": "google_workspace_assets",
        "table": "bronze_google_workspace_assets",
        "renames": [("assetId", "asset_id"), ("assetType", "asset_type")],
        "columns": ["asset_id", "name", "asset_type", "environment", "sensitivity"],
        "schema": [
            T.StructField("asset_id", T.StringType(), True),
            T.StructField("name", T.StringType(), True),
            T.StructField("asset_type", T.StringType(), True),
            T.StructField("environment", T.StringType(), True),
            T.StructField("sensitivity", T.StringType(), True),
        ],
    },
    {
        "path": "google_workspace/access_edges.jsonl",
        "src_system": "google_workspace_seed",
        "src_object": "google_workspace_access_edges",
        "table": "bronze_google_workspace_access_edges",
        "renames": [("principalId", "principal_id"), ("assetId", "asset_id")],
        "columns": ["principal_id", "asset_id", "relationship", "permission", "status"],
        "schema": [
            T.StructField("principal_id", T.StringType(), True),
            T.StructField("asset_id", T.StringType(), True),
            T.StructField("relationship", T.StringType(), True),
            T.StructField("permission", T.StringType(), True),
            T.StructField("status", T.StringType(), True),
        ],
    },
    {
        "path": "slack/users.jsonl",
        "src_system": "slack_seed",
        "src_object": "slack_users",
        "table": "bronze_slack_users",
        "renames": [("userId", "user_id"), ("displayName", "display_name"), ("employmentStatus", "employment_status")],
        "dates": [("terminationDate", "termination_date")],
        "columns": ["user_id", "email", "display_name", "status", "employment_status", "termination_date"],
        "schema": [
            T.StructField("user_id", T.StringType(), True),
            T.StructField("email", T.StringType(), True),
            T.StructField("display_name", T.StringType(), True),
            T.StructField("status", T.StringType(), True),
            T.StructField("employment_status", T.StringType(), True),
            T.StructField("termination_date", T.DateType(), True),
        ],
    },
    {
        "path": "slack/channels.jsonl",
        "src_system": "slack_seed",
        "src_object": "slack_channels",
        "table": "bronze_slack_channels",
        "renames": [("channelId", "channel_id"), ("channelType", "channel_type")],
        "columns": ["channel_id", "name", "channel_type", "environment", "sensitivity"],
        "schema": [
            T.StructField("channel_id", T.StringType(), True),
            T.StructField("name", T.StringType(), True),
            T.StructField("channel_type", T.StringType(), True),
            T.StructField("environment", T.StringType(), True),
            T.StructField("sensitivity", T.StringType(), True),
        ],
    },
    {
        "path": "slack/access_edges.jsonl",
        "src_system": "slack_seed",
        "src_object": "slack_access_edges",
        "table": "bronze_slack_access_edges",
        "renames": [("principalId", "principal_id"), ("assetId", "asset_id")],
        "columns": ["principal_id", "asset_id", "relationship", "permission", "status"],
        "schema": [
            T.StructField("principal_id", T.StringType(), True),
            T.StructField("asset_id", T.StringType(), True),
            T.StructField("relationship", T.StringType(), True),
            T.StructField("permission", T.StringType(), True),
            T.StructField("status", T.StringType(), True),
        ],
    },
]

for cfg in SAAS_BRONZE_SOURCES:
    path = f"{RAW_BASE}/{cfg['path']}"
    table_name = cfg["table"]
    if raw_file_exists(path):
        df = spark.read.json(path)
        df = add_meta(df, cfg["src_system"], cfg["src_object"], path)
        df = transform_source(
            df,
            renames=cfg.get("renames"),
            timestamps=cfg.get("timestamps"),
            dates=cfg.get("dates"),
        )
        (df.select(
            "_src_system", "_src_object", "_src_file_path", "_ingestion_ts",
            "_run_id", "_batch_id", "_checksum", "_raw_payload", "_schema_version",
            *cfg["columns"],
        ).write.format("delta").mode("overwrite")
         .saveAsTable(f"{CATALOG}.bronze.{table_name}"))
    else:
        write_empty_bronze_table(table_name, cfg["schema"])
        print(f"{table_name}: source file missing, wrote empty table")
        continue

    count = spark.table(f"{CATALOG}.bronze.{table_name}").count()
    print(f"{table_name}: {count} rows")


In [0]:
required_bronze_tables = [
    "bronze_azure_applications",
    "bronze_azure_application_owners",
    "bronze_azure_password_credentials",
    "bronze_identity_lifecycle_seed",
    "bronze_assets_seed",
    "bronze_access_edges_seed",
]

optional_bronze_tables = [
    "bronze_github_apps",
    "bronze_github_credentials",
    "bronze_github_repos",
    "bronze_github_access_edges",
    "bronze_google_workspace_users",
    "bronze_google_workspace_assets",
    "bronze_google_workspace_access_edges",
    "bronze_slack_users",
    "bronze_slack_channels",
    "bronze_slack_access_edges",
]

summary = ["ingest smoke test passed"]
for table in required_bronze_tables:
    actual = spark.table(f"{CATALOG}.bronze.{table}").count()
    assert actual > 0, f"{table} has no rows"
    line = f"{table}: {actual} rows"
    print(line)
    summary.append(line)

for table in optional_bronze_tables:
    actual = spark.table(f"{CATALOG}.bronze.{table}").count()
    line = f"{table}: {actual} rows"
    print(line)
    summary.append(line)

print("ingest complete")
dbutils.notebook.exit("\n".join(summary))
